In [2]:
# Step 2: RAG Pipeline
# Install required packages first if needed:
# pip install llama-index chromadb llama-index-vector-stores-chroma

import os
from dotenv import load_dotenv
import json
import chromadb
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

load_dotenv()
print("Libraries loaded!")

Libraries loaded!


In [3]:
# Load enriched data
with open('../data/pubmedqa_entities.json', 'r') as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")
print(f"Sample keys: {list(records[0].keys())}")

Loaded 1000 records
Sample keys: ['pubid', 'question', 'contexts', 'long_answer', 'final_decision', 'entities']


In [4]:
# Create ChromaDB vector store
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")
chroma_collection = chroma_client.get_or_create_collection("pubmedqa")

print(f"ChromaDB collection created: {chroma_collection.name}")

ChromaDB collection created: pubmedqa


In [5]:
# Convert records to LlamaIndex Documents
documents = []
for record in records:
    # Combine all contexts into one text
    full_text = " ".join(record['contexts'])
    
    doc = Document(
        text=full_text,
        metadata={
            'pubid': str(record['pubid']),
            'question': record['question'],
            'final_decision': record['final_decision'],
            'long_answer': record['long_answer']
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")

Created 1000 documents


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

# Use HuggingFace embedding model
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model
Settings.llm = None  # No LLM for now

print("Embedding model loaded!")